# Volatility Dynamics 08 — jump arrivals as a point process

**VD 017.** Modules 03–07 showed that raw q99 *variance* events are explained by the volatility level. Module 08 studies **jumps**: returns extreme *relative to* causal local volatility, and asks whether they arrive as a Poisson process, follow the clock/volatility, or **self-excite** (Hawkes).

**Jump:** $\tilde r_t = r_t/s_a(\text{time of day})$ (smooth 2-harmonic clock, training days); $BV_t$ = **two-sided** jump-robust bipower over the hour before and after $t$ (pairs touching $t$ excluded); $z_t=|\tilde r_t|/\sqrt{BV_t}$; $J_t=\mathbf 1\{z_t>c\}$, **$c=4$** (sensitivity grid 3–5 reported, not used for the decision).

**Models** (5-min bars, $P(J=1)=1-e^{-\lambda}$):
(a) $\lambda=e^{\alpha_a}$ · (b) $\lambda=\exp(\alpha_a+\text{Fourier}_2+\gamma\log\sigma^{local})$ · (c) (b) $+\ \eta_s E^{self}+\eta_c E^{cross}$, exponential kernel, half-life profiled. Branching ratio $\rho=\eta_s+\eta_c(N-1)$.

**Pre-registered decision:** 5-fold blocked CV by day blocks. Self-excitation iff out-of-fold logLik/row (c)−(b) > 0 with day-bootstrap q2.5 > 0 and ≥ 50 scored jumps; same rule (b)−(a) for inhomogeneity. Chronological 70/30 holdout reported as secondary. Validated on synthetic data of this size: 0/8 false passes under a null with a common volatility factor and Student-t(4) diffusion; power ≈ 1/3 against a true $\rho=0.37$, with $\rho$ recovered (0.23–0.43). **A FAIL at this sample size is not evidence of absence.** Two-sided local volatility makes this an *explanatory* model (does a past jump raise the jump probability beyond local volatility?), not a real-time forecaster.

Inputs: `state_time`, `underlying`, `return_5m` only. Outputs: `08/`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_jump_hawkes as jh
START = os.getenv('VD8_START', '').strip() or None
END = os.getenv('VD8_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD8_ASSETS', '').split(',') if x.strip()]
N_BOOT = int(os.getenv('VD8_N_BOOT', str(jh.N_BOOT)))
N_BOOT_SENS = int(os.getenv('VD8_N_BOOT_SENS', '500'))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '08'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 08', jh.VERSION)
print('selftest:', jh.selftest())


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

df = load_frame([], START, END, ASSETS)
grid = vb.build_grid(df)
panel = jh.build_panel(grid)
train_rows, test_rows, test_first_day = jh.chronological_rows(panel)
print('assets', panel.assets, '| bars', panel.T, '| holdout starts', pd.Timestamp(test_first_day * 288 * vb.BAR_NS, tz='UTC'))
S = jh.clock_factors(panel, train_rows)
Z, BV = jh.jump_statistics(panel, S)
C = jh.baseline_covariates(panel, S, BV)
fig, ax = plt.subplots(figsize=(8, 3))
for a, name in enumerate(panel.assets):
    ax.plot(np.arange(288) / 12, S[:, a], label=name)
ax.set_xlabel('UTC hour'); ax.set_ylabel('clock factor'); ax.legend(ncol=4); plt.show()


In [ ]:
J, valid = jh.jump_indicator(Z, jh.PRIMARY_C)
per_asset, by_hour, cojump = jh.jump_descriptives(panel, J, valid, Z, jh.PRIMARY_C)
display(per_asset.round(3)); display(cojump)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(by_hour.hour_utc, by_hour.jump_rate_per_1000_bars); ax.set_xlabel('UTC hour'); ax.set_ylabel('jumps / 1000 bars'); plt.show()
z = Z[np.isfinite(Z)]
print('z tail shares:', {c: float(np.mean(z > c)) for c in jh.C_GRID})


In [ ]:
t0 = time.time()
primary = jh.analyse(panel, J, valid, C, train_rows, test_rows, n_boot=N_BOOT, label='jumps_c4')
print(f'primary analysis {time.time()-t0:.0f}s')
print(json.dumps(primary['summary'], indent=2, default=float))
display(primary['cv_scores']); display(primary['cv_boot'].round(6)); display(primary['cv_folds'].round(4))
display(primary['holdout_boot'].round(6)); display(primary['gof'].round(4))
display(primary['params'].round(4))
prof = primary['profile']
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(prof.half_life_min, prof.train_loglik, marker='o'); ax.set_xscale('log')
ax.set_xlabel('kernel half-life (min)'); ax.set_ylabel('full-sample logLik'); plt.show()


In [ ]:
sens_rows = []
for c in jh.C_GRID:
    Jc, vc = jh.jump_indicator(Z, c)
    r = jh.analyse(panel, Jc, vc, C, train_rows, test_rows, n_boot=N_BOOT_SENS, label=f'jumps_c{c}')
    s = r['summary']
    sens_rows.append({
        'c': c, 'is_primary': c == jh.PRIMARY_C, 'n_jumps': s['n_events_full_sample'],
        'branching_ratio': s['branching_ratio'], 'half_life_min': s['hawkes_half_life_min_full_sample'],
        'eta_self': s['eta_self'], 'eta_cross': s['eta_cross'],
        'cv_self_excitation': s['primary_blocked_cv']['self_excitation_c_vs_b']['status'],
        'cv_delta_c_vs_b': s['primary_blocked_cv']['self_excitation_c_vs_b']['delta_loglik_per_row'],
        'cv_inhomogeneity': s['primary_blocked_cv']['inhomogeneity_b_vs_a']['status'],
    })
sensitivity = pd.DataFrame(sens_rows)
display(sensitivity.round(5))


In [ ]:
Jq, vq = jh.q99_indicator(panel)
secondary_q99 = jh.analyse(panel, Jq, vq, C, train_rows, test_rows, n_boot=N_BOOT_SENS, label='raw_q99_onsets')
print(json.dumps(secondary_q99['summary'], indent=2, default=float))


In [ ]:
per_asset.to_csv(OUT / '08_jumps_per_asset.csv', index=False)
by_hour.to_csv(OUT / '08_jump_rate_by_hour.csv', index=False)
cojump.to_csv(OUT / '08_cojump_counts.csv', index=False)
pd.DataFrame(S, columns=panel.assets).assign(slot=np.arange(288)).to_csv(OUT / '08_clock_factors.csv', index=False)
for k in ['cv_scores', 'cv_boot', 'cv_folds', 'holdout_scores', 'holdout_boot', 'gof', 'params', 'profile']:
    primary[k].to_csv(OUT / f'08_primary_{k}.csv', index=False)
for k in ['cv_boot', 'cv_folds', 'gof', 'params']:
    secondary_q99[k].to_csv(OUT / f'08_q99_{k}.csv', index=False)
sensitivity.to_csv(OUT / '08_c_sensitivity.csv', index=False)
(OUT / '08_decision.json').write_text(json.dumps({'primary_c4': primary['summary'], 'secondary_raw_q99': secondary_q99['summary']}, indent=2, default=float))
meta = {
    'version': jh.VERSION, 'module': '08_jump_point_process', 'primary_c': jh.PRIMARY_C, 'c_grid': list(jh.C_GRID),
    'local_vol': 'two-sided bipower, +/- %d bars, pairs touching t excluded' % jh.LOCAL_HALF_WINDOW, 'half_life_grid_bars': list(jh.HALF_LIFE_GRID_BARS),
    'decision_primary': 'blocked 5-fold CV by day; delta logLik/row (c)-(b) > 0 with day-bootstrap q2.5 > 0, >= 50 jumps',
    'holdout_first_day': str(pd.Timestamp(test_first_day * 288 * vb.BAR_NS, tz='UTC')), 'n_boot': N_BOOT,
    'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max()), 'n_rows': int(len(df)),
}
(OUT / '08_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote', OUT)
